# 0. Import

In [61]:
!pip install wandb -q

import warnings
warnings.filterwarnings('ignore')


In [31]:
import json
import wandb
import pandas as pd
import torch

from torch.utils.data import Dataset, DataLoader
import json
import numpy as np
from pathlib import Path

import torch
import torch.nn as nn

In [32]:
import os
import wandb
from kaggle_secrets import UserSecretsClient

# Fetch the secret token safely
user_secrets = UserSecretsClient()
os.environ["WANDB_API_KEY"] = user_secrets.get_secret("WANDB_API_KEY")

# Log into WandB
wandb.login()

True

# 1. Making the Modality

In [33]:
def load_skeleton(trial_path):

    prediction_dir = Path(trial_path) / "predictions"

    json_files = sorted(prediction_dir.glob("*.json"))

    frames = []

    for json_file in json_files:

        with open(json_file, "r") as f:
            data = json.load(f)

        # One person per frame based on our inspection
        person = data[0]

        keypoints = np.asarray(
            person["keypoints"],
            dtype=np.float32
        )

        frames.append(keypoints)

    if len(frames) == 0:
        return None

    return np.stack(frames)

## 1. Building the training index

In [34]:
def temporal_resample(x, target_frames=64):

    T = x.shape[0]

    if T == target_frames:
        return x.astype(np.float32)

    old_indices = np.linspace(
        0,
        T - 1,
        T
    )

    new_indices = np.linspace(
        0,
        T - 1,
        target_frames
    )

    output = np.empty(
        (target_frames, x.shape[1], x.shape[2]),
        dtype=np.float32
    )

    for joint in range(x.shape[1]):
        for coord in range(x.shape[2]):

            output[:, joint, coord] = np.interp(
                new_indices,
                old_indices,
                x[:, joint, coord]
            )

    return output

In [35]:
def normalize_skeleton(x):
    """
    x: (T, 17, 3)

    Makes skeleton coordinates root-relative.
    Joint 0 is treated as the root.
    """

    x = x.copy()

    # Root-relative coordinates
    root = x[:, 0:1, :]
    x = x - root

    return x.astype(np.float32)

def temporal_resample(x, target_frames=64):
    """
    Resample the complete sequence to exactly target_frames.
    """

    T = x.shape[0]

    if T == target_frames:
        return x.astype(np.float32)

    if T == 1:
        return np.repeat(
            x,
            target_frames,
            axis=0
        ).astype(np.float32)

    old_indices = np.linspace(
        0,
        T - 1,
        T
    )

    new_indices = np.linspace(
        0,
        T - 1,
        target_frames
    )

    output = np.empty(
        (target_frames, x.shape[1], x.shape[2]),
        dtype=np.float32
    )

    for joint in range(x.shape[1]):

        for coord in range(x.shape[2]):

            output[:, joint, coord] = np.interp(
                new_indices,
                old_indices,
                x[:, joint, coord]
            )

    return output.astype(np.float32)

def get_bone_vectors(self, x):
    """
    x: (T, 17, 3)

    Returns:
        bone_vectors: (T, 17, 3)
    """

    # COCO-17 skeleton hierarchy
    #
    # 0  nose
    # 1  left eye
    # 2  right eye
    # 3  left ear
    # 4  right ear
    # 5  left shoulder
    # 6  right shoulder
    # 7  left elbow
    # 8  right elbow
    # 9  left wrist
    # 10 right wrist
    # 11 left hip
    # 12 right hip
    # 13 left knee
    # 14 right knee
    # 15 left ankle
    # 16 right ankle

    parents = [
        -1,  # nose
         0,  # left eye
         0,  # right eye
         1,  # left ear
         2,  # right ear
        11,  # left shoulder -> left hip
        12,  # right shoulder -> right hip
         5,  # left elbow
         6,  # right elbow
         7,  # left wrist
         8,  # right wrist
        -1,  # left hip
        -1,  # right hip
        11,  # left knee
        12,  # right knee
        13,  # left ankle
        14   # right ankle
    ]

    bone_vectors = np.zeros_like(x)

    for joint, parent in enumerate(parents):

        if parent != -1:

            bone_vectors[:, joint, :] = (
                x[:, joint, :] -
                x[:, parent, :]
            )

    return bone_vectors

## 3. Dataset Class

In [36]:
class SkeletonDataset(Dataset):
    def __init__(self, df, sequence_length=64):
        self.df = df.reset_index(drop=True)
        self.sequence_length = sequence_length

    def load_skeleton(self, path):
        prediction_dir = Path(path) / "predictions"
        json_files = sorted(prediction_dir.glob("*.json"))

        keypoint_frames = []
        confidence_frames = []

        for json_file in json_files:
            with open(json_file, "r") as f:
                data = json.load(f)

            if len(data) == 0:
                continue

            person = data[0]

            # -------------------------
            # Keypoints
            # -------------------------
            keypoints = np.asarray(
                person["keypoints"],
                dtype=np.float32
            )

            # Expected: (17, 3)
            if keypoints.shape != (17, 3):
                continue

            # -------------------------
            # Confidence scores
            # -------------------------
            scores = np.asarray(
                person["keypoint_scores"],
                dtype=np.float32
            ).reshape(-1)

            # Expected: 17 scores
            if scores.shape[0] != 17:
                continue

            scores = np.clip(scores, 0.0, 1.0)

            keypoint_frames.append(keypoints)
            confidence_frames.append(scores)

        if len(keypoint_frames) == 0:
            return None, None

        keypoints = np.stack(keypoint_frames)       # (T,17,3)
        scores = np.stack(confidence_frames)        # (T,17)

        return keypoints, scores

    # --------------------------------------------------
    # Pelvis-centered + scale normalization
    # --------------------------------------------------
    def normalize_skeleton(self, x):
        x = x.copy()

        # COCO-17
        # left hip  = 11
        # right hip = 12
        # left shoulder  = 5
        # right shoulder = 6

        pelvis = (
            x[:, 11:12, :] +
            x[:, 12:13, :]
        ) / 2.0

        x = x - pelvis

        shoulder_center = (
            x[:, 5:6, :] +
            x[:, 6:7, :]
        ) / 2.0

        scale = np.linalg.norm(
            shoulder_center,
            axis=2,
            keepdims=True
        )

        scale = np.maximum(scale, 1e-6)

        x = x / scale

        return x

    # --------------------------------------------------
    # Bone vectors
    # --------------------------------------------------
    def get_bone_vectors(self, x):

        parents = [
            -1, 0, 0, 1, 2,
            11, 12,
            5, 6,
            7, 8,
            -1, -1,
            11, 12,
            13, 14
        ]

        bones = np.zeros_like(x)

        for j, p in enumerate(parents):
            if p >= 0:
                bones[:, j, :] = x[:, j, :] - x[:, p, :]

        return bones

    # --------------------------------------------------
    # Temporal resampling
    # --------------------------------------------------
    def temporal_resample(self, x):

        T = x.shape[0]

        if T == self.sequence_length:
            return x.astype(np.float32)

        if T == 1:
            return np.repeat(
                x,
                self.sequence_length,
                axis=0
            ).astype(np.float32)

        old_indices = np.linspace(0, T - 1, T)
        new_indices = np.linspace(
            0,
            T - 1,
            self.sequence_length
        )

        output = np.empty(
            (
                self.sequence_length,
                x.shape[1],
                x.shape[2]
            ),
            dtype=np.float32
        )

        for joint in range(x.shape[1]):
            for coord in range(x.shape[2]):
                output[:, joint, coord] = np.interp(
                    new_indices,
                    old_indices,
                    x[:, joint, coord]
                )

        return output

    # --------------------------------------------------
    # Dataset
    # --------------------------------------------------
    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):

        row = self.df.iloc[idx]

        skeleton, confidence = self.load_skeleton(
            row["skeleton"]
        )

        # Missing skeleton
        if skeleton is None:

            skeleton = np.zeros(
                (self.sequence_length, 17, 3),
                dtype=np.float32
            )

            confidence = np.zeros(
                (self.sequence_length, 17),
                dtype=np.float32
            )

        # -------------------------
        # Normalize coordinates
        # -------------------------
        skeleton = self.normalize_skeleton(skeleton)

        # -------------------------
        # Velocity
        # -------------------------
        velocity = np.diff(
            skeleton,
            axis=0,
            prepend=skeleton[0:1]
        )

        # -------------------------
        # Acceleration
        # -------------------------
        acceleration = np.diff(
            velocity,
            axis=0,
            prepend=velocity[0:1]
        )

        # -------------------------
        # Bone vectors
        # -------------------------
        bones = self.get_bone_vectors(skeleton)

        # -------------------------
        # Add confidence
        #
        # XYZ        = 3
        # velocity   = 3
        # acceleration = 3
        # bones      = 3
        # confidence = 1
        #
        # TOTAL = 13 features/joint
        # -------------------------


        features = np.concatenate(
            [
                skeleton,
                velocity,
                acceleration,
                bones,
            ],
            axis=2
        )

        # (T, 17, 13)

        features = self.temporal_resample(features)

        # Final shape:
        # (64, 17, 13)

        X = torch.tensor(
            features,
            dtype=torch.float32
        )

        y = torch.tensor(
            row["label"],
            dtype=torch.long
        )

        return X, y

In [37]:
from pathlib import Path
import pandas as pd

TRAIN_DATA = Path(
    "/kaggle/input/datasets/samasiayushman/"
    "small-model-track/Training/Training/data"
)

MODALITIES = [
    "Skeleton",
    "Depth_Color",
    "IR",
    "Thermal",
    "IMU",
    "Radar",
]

# ---------------------------------------------------------
# Build trial index from Skeleton
# ---------------------------------------------------------

records = []

skeleton_root = TRAIN_DATA / "Skeleton"

for action_dir in sorted(skeleton_root.iterdir()):

    if not action_dir.is_dir():
        continue

    action_name = action_dir.name
    label = int(action_name.split("_")[0])

    for user_dir in sorted(action_dir.iterdir()):

        if not user_dir.is_dir():
            continue

        user = user_dir.name

        for trial_dir in sorted(user_dir.iterdir()):

            if not trial_dir.is_dir():
                continue

            records.append({
                "action": action_name,
                "label": label,
                "user": user,
                "trial": trial_dir.name,
                "path": str(trial_dir),
            })


train_df = pd.DataFrame(records)

print("Training trials:", len(train_df))
print("Classes:", train_df["label"].nunique())
print("Users:", train_df["user"].nunique())

train_df.head()

Training trials: 2931
Classes: 40
Users: 18


,action,label,user,trial,path
0,0_Wash_face,0,user16,1-1-1,/kaggle/input/datasets/samasiayushman/small-mo...
1,0_Wash_face,0,user16,1-1-2,/kaggle/input/datasets/samasiayushman/small-mo...
2,0_Wash_face,0,user16,1-1-3,/kaggle/input/datasets/samasiayushman/small-mo...
3,0_Wash_face,0,user18,7-1-1,/kaggle/input/datasets/samasiayushman/small-mo...
4,0_Wash_face,0,user18,7-1-2,/kaggle/input/datasets/samasiayushman/small-mo...


In [38]:
from sklearn.model_selection import GroupShuffleSplit

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.2,
    random_state=42
)

train_idx, val_idx = next(
    splitter.split(
        train_df,
        train_df["label"],
        groups=train_df["user"]
    )
)

train_split = train_df.iloc[train_idx].reset_index(drop=True)
val_split = train_df.iloc[val_idx].reset_index(drop=True)

print("Train:", len(train_split))
print("Validation:", len(val_split))

print("Train users:")
print(sorted(train_split["user"].unique()))

print("\nValidation users:")
print(sorted(val_split["user"].unique()))

Train: 2238
Validation: 693
Train users:
['user17', 'user18', 'user19', 'user20', 'user21', 'user23', 'user24', 'user3', 'user4', 'user5', 'user6', 'user7', 'user8', 'user9']

Validation users:
['user1', 'user16', 'user2', 'user22']


In [39]:
train_dataset = SkeletonDataset(
    train_split.assign(skeleton=train_split["path"]),
    sequence_length=64
)

val_dataset = SkeletonDataset(
    val_split.assign(skeleton=val_split["path"]),
    sequence_length=64
)

print("Train dataset:", len(train_dataset))
print("Val dataset:", len(val_dataset))

Train dataset: 2238
Val dataset: 693


In [40]:
train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=0,
    pin_memory=False
)

val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=0,
    pin_memory=False
)

In [41]:
X, y = next(iter(train_loader))

print("Train batch X:", X.shape)
print("Train batch y:", y.shape)

X, y = next(iter(val_loader))

print("Val batch X:", X.shape)
print("Val batch y:", y.shape)

Train batch X: torch.Size([32, 64, 17, 12])
Train batch y: torch.Size([32])
Val batch X: torch.Size([32, 64, 17, 12])
Val batch y: torch.Size([32])


In [42]:
dataset =  SkeletonDataset(
    train_split.assign(skeleton=train_split["path"]),
    sequence_length=64
)


X, y = dataset[0]

print("X shape:", X.shape)
print("y:", y)
print("dtype:", X.dtype)

print("Min:", X.min().item())
print("Max:", X.max().item())
print("Mean:", X.mean().item())
print("Std:", X.std().item())

X shape: torch.Size([64, 17, 12])
y: tensor(0)
dtype: torch.float32
Min: -1.2548960447311401
Max: 1.66280198097229
Mean: -0.0242230873554945
Std: 0.29696953296661377


---

In [43]:
print("Root joint mean:", X[:, 0, :].abs().mean().item())

print( "Root joint max:", X[:, 0, :].abs().max().item())

Root joint mean: 0.10289439558982849
Root joint max: 0.6045839190483093


# A. Thermal

In [44]:
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from PIL import Image

BASE_DATA_ROOT = Path(
    "/kaggle/input/datasets/samasiayushman/"
    "small-model-track/Training/Training/data"
)
THERMAL_ROOT = BASE_DATA_ROOT / "Thermal"

THERMAL_SEQUENCE_LENGTH = 32
THERMAL_IMAGE_SIZE = 112

VAL_USERS = {"user8", "user9", "user23", "user24"}


def load_thermal_trial(trial_dir):
    """
    Load all thermal frames from one trial in chronological order.
    """
    
    files = sorted(
        trial_dir.glob("*.jpg"),
        key=lambda p: int(p.stem.split("_")[-1])
    )
    
    if len(files) == 0:
        return []
    
    return files


def resample_frame_indices(n_frames, target_length=THERMAL_SEQUENCE_LENGTH):
    """
    Select evenly spaced frame indices.
    """
    
    if n_frames == 0:
        return []
    
    if n_frames == target_length:
        return np.arange(n_frames)
    
    if n_frames == 1:
        return np.zeros(target_length, dtype=int)
    
    return np.linspace(
        0,
        n_frames - 1,
        target_length
    ).round().astype(int)


class ThermalDataset(Dataset):

    def __init__(
        self,
        df,
        action_to_idx,
        thermal_root,
        image_size=THERMAL_IMAGE_SIZE,
        sequence_length=THERMAL_SEQUENCE_LENGTH
    ):
        
        self.df = df.reset_index(drop=True)
        self.action_to_idx = action_to_idx
        self.thermal_root = Path(thermal_root)
        self.image_size = image_size
        self.sequence_length = sequence_length

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):

        row = self.df.iloc[idx]

        trial_dir = (
            self.thermal_root
            / row["action"]
            / row["user"]
            / row["trial"]
        )

        frame_files = load_thermal_trial(trial_dir)

        if len(frame_files) == 0:
            # Should rarely happen, but keep tensor shape valid.
            frames = np.zeros(
                (
                    self.sequence_length,
                    3,
                    self.image_size,
                    self.image_size
                ),
                dtype=np.float32
            )

        else:

            indices = resample_frame_indices(
                len(frame_files),
                self.sequence_length
            )

            frame_list = []

            for frame_idx in indices:

                img = Image.open(frame_files[frame_idx]).convert("RGB")

                img = img.resize(
                    (self.image_size, self.image_size),
                    Image.BILINEAR
                )

                arr = np.asarray(
                    img,
                    dtype=np.float32
                ) / 255.0

                # H,W,C → C,H,W
                arr = np.transpose(arr, (2, 0, 1))

                frame_list.append(arr)

            frames = np.stack(frame_list, axis=0)

        X = torch.tensor(
            frames,
            dtype=torch.float32
        )

        y = torch.tensor(
            self.action_to_idx[row["action"]],
            dtype=torch.long
        )

        return X, y

In [45]:
# ============================================================
# THERMAL TRAIN / VALIDATION SPLIT
# ============================================================

thermal_train_df = train_df[
    ~train_df["user"].isin(VAL_USERS)
].reset_index(drop=True)

thermal_val_df = train_df[
    train_df["user"].isin(VAL_USERS)
].reset_index(drop=True)

print("Thermal train:", len(thermal_train_df))
print("Thermal val:", len(thermal_val_df))

print(
    "Train users:",
    sorted(thermal_train_df["user"].unique())
)

print(
    "Val users:",
    sorted(thermal_val_df["user"].unique())
)

Thermal train: 2295
Thermal val: 636
Train users: ['user1', 'user16', 'user17', 'user18', 'user19', 'user2', 'user20', 'user21', 'user22', 'user3', 'user4', 'user5', 'user6', 'user7']
Val users: ['user23', 'user24', 'user8', 'user9']


In [46]:
# ============================================================
# ACTION → CLASS INDEX
# ============================================================

actions = sorted(train_df["action"].unique())

action_to_idx = {
    action: idx
    for idx, action in enumerate(actions)
}

idx_to_action = {
    idx: action
    for action, idx in action_to_idx.items()
}

print("Number of actions:", len(action_to_idx))
print("\nFirst 10 classes:")

for idx in range(min(10, len(idx_to_action))):
    print(idx, "→", idx_to_action[idx])

Number of actions: 40

First 10 classes:
0 → 0_Wash_face
1 → 10_Stir_drinks
2 → 11_Peel_fruits
3 → 12_Sweep_the_floor
4 → 13_Mop_the_floor
5 → 14_Wipe_bowls
6 → 15_Wipe_windows_and_tables
7 → 16_Fold_clothes
8 → 17_Tap_the_keyboard
9 → 18_Write


In [47]:
# CREATE DATA LOADERS

thermal_train_dataset = ThermalDataset(
    thermal_train_df,
    action_to_idx,
    THERMAL_ROOT
)

thermal_val_dataset = ThermalDataset(
    thermal_val_df,
    action_to_idx,
    THERMAL_ROOT
)

thermal_train_loader = DataLoader(
    thermal_train_dataset,
    batch_size=16,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

thermal_val_loader = DataLoader(
    thermal_val_dataset,
    batch_size=16,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

In [48]:
X, y = next(iter(thermal_train_loader))

print("X shape:", X.shape)
print("y shape:", y.shape)

print("X dtype:", X.dtype)
print("X min:", X.min().item())
print("X max:", X.max().item())
print("X mean:", X.mean().item())
print("X std:", X.std().item())
print("NaN:", torch.isnan(X).any().item())
print("Inf:", torch.isinf(X).any().item())

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


X shape: torch.Size([16, 32, 3, 112, 112])
y shape: torch.Size([16])
X dtype: torch.float32
X min: 0.0
X max: 1.0
X mean: 0.4114123582839966
X std: 0.3070380985736847
NaN: False
Inf: False


# T1 — THERMAL CNN + BiLSTM + TEMPORAL ATTENTION


In [54]:
# ============================================================
# T2 THERMAL DATASET
# ============================================================

class ThermalDataset_T2(Dataset):

    def __init__(
        self,
        df,
        action_to_idx,
        thermal_root,
        width=THERMAL_T2_WIDTH,
        height=THERMAL_T2_HEIGHT,
        sequence_length=THERMAL_SEQUENCE_LENGTH
    ):

        self.df = df.reset_index(drop=True)
        self.action_to_idx = action_to_idx
        self.thermal_root = Path(thermal_root)

        self.width = width
        self.height = height
        self.sequence_length = sequence_length

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):

        row = self.df.iloc[idx]

        trial_dir = (
            self.thermal_root
            / row["action"]
            / row["user"]
            / row["trial"]
        )

        frame_files = load_thermal_trial(trial_dir)

        if len(frame_files) == 0:

            frames = np.zeros(
                (
                    self.sequence_length,
                    3,
                    self.height,
                    self.width
                ),
                dtype=np.float32
            )

        else:

            indices = resample_frame_indices(
                len(frame_files),
                self.sequence_length
            )

            frame_list = []

            for frame_idx in indices:

                img = Image.open(
                    frame_files[frame_idx]
                ).convert("RGB")

                img = img.resize(
                    (self.width, self.height),
                    Image.BILINEAR
                )

                arr = np.asarray(
                    img,
                    dtype=np.float32
                ) / 255.0

                arr = np.transpose(
                    arr,
                    (2, 0, 1)
                )

                frame_list.append(arr)

            frames = np.stack(
                frame_list,
                axis=0
            )

        X = torch.tensor(
            frames,
            dtype=torch.float32
        )

        y = torch.tensor(
            self.action_to_idx[row["action"]],
            dtype=torch.long
        )

        return X, y

In [55]:
THERMAL_T2_WIDTH = 160
THERMAL_T2_HEIGHT = 120


class ThermalCNN_T2(nn.Module):

    def __init__(self, feature_dim=256):
        super().__init__()

        self.features = nn.Sequential(

            # 160x120 -> 80x60
            nn.Conv2d(3, 32, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(32),
            nn.GELU(),

            # 80x60 -> 40x30
            nn.Conv2d(32, 64, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(64),
            nn.GELU(),

            # 40x30 -> 20x15
            nn.Conv2d(64, 128, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(128),
            nn.GELU(),

            # 20x15 -> 10x8
            nn.Conv2d(128, 192, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(192),
            nn.GELU(),

            # 10x8 -> 5x4
            nn.Conv2d(192, 256, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(256),
            nn.GELU(),

            nn.AdaptiveAvgPool2d(1)
        )

        self.projection = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256, feature_dim),
            nn.LayerNorm(feature_dim),
            nn.GELU()
        )

    def forward(self, x):
        x = self.features(x)
        return self.projection(x)


class ThermalT2(nn.Module):

    def __init__(
        self,
        num_classes=40,
        frame_feature_dim=256,
        lstm_hidden=128
    ):
        super().__init__()

        self.cnn = ThermalCNN_T2(
            feature_dim=frame_feature_dim
        )

        self.lstm = nn.LSTM(
            input_size=frame_feature_dim,
            hidden_size=lstm_hidden,
            num_layers=2,
            batch_first=True,
            bidirectional=True,
            dropout=0.2
        )

        temporal_dim = lstm_hidden * 2

        self.temporal_attention = nn.MultiheadAttention(
            embed_dim=temporal_dim,
            num_heads=4,
            dropout=0.1,
            batch_first=True
        )

        self.norm = nn.LayerNorm(temporal_dim)

        self.frame_attention = nn.Sequential(
            nn.Linear(temporal_dim, 64),
            nn.Tanh(),
            nn.Linear(64, 1)
        )

        self.classifier = nn.Sequential(
            nn.Linear(temporal_dim, 128),
            nn.GELU(),
            nn.Dropout(0.3),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):

        B, T, C, H, W = x.shape

        # CNN processes frames independently
        x = x.reshape(B * T, C, H, W)

        frame_features = self.cnn(x)

        frame_features = frame_features.reshape(
            B, T, -1
        )

        # Temporal modeling
        temporal, _ = self.lstm(frame_features)

        # Temporal self-attention
        attended, _ = self.temporal_attention(
            temporal,
            temporal,
            temporal
        )

        temporal = self.norm(
            temporal + attended
        )

        # Learned temporal pooling
        scores = self.frame_attention(
            temporal
        ).squeeze(-1)

        weights = torch.softmax(
            scores,
            dim=1
        )

        pooled = torch.sum(
            temporal * weights.unsqueeze(-1),
            dim=1
        )

        return self.classifier(pooled)

In [57]:
# ============================================================
# T2 DATASETS / LOADERS
# ============================================================

thermal_T2_train_dataset = ThermalDataset_T2(
    thermal_train_df,
    action_to_idx,
    THERMAL_ROOT
)

thermal_T2_val_dataset = ThermalDataset_T2(
    thermal_val_df,
    action_to_idx,
    THERMAL_ROOT
)

thermal_T2_train_loader = DataLoader(
    thermal_T2_train_dataset,
    batch_size=16,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

thermal_T2_val_loader = DataLoader(
    thermal_T2_val_dataset,
    batch_size=16,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

# ============================================================
# T2 MODEL CHECK
# ============================================================

model = ThermalT2(
    num_classes=40,
    frame_feature_dim=256,
    lstm_hidden=128
).to(device)

num_params = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

model_size_mb = (
    num_params * 4 / (1024 ** 2)
)

print("Trainable parameters:", f"{num_params:,}")
print("Approx FP32 size:", f"{model_size_mb:.2f} MB")

X, y = next(iter(thermal_T2_train_loader))

print("Input:", X.shape)

X = X.to(device)

with torch.no_grad():
    output = model(X)

print("Output:", output.shape)

Trainable parameters: 1,933,673
Approx FP32 size: 7.38 MB
Input: torch.Size([16, 32, 3, 120, 160])
Output: torch.Size([16, 40])


In [58]:
# ============================================================
# T2 — FRESH TRAINING SETUP
# ============================================================

model = ThermalT2(
    num_classes=40,
    frame_feature_dim=256,
    lstm_hidden=128
).to(device)

criterion = nn.CrossEntropyLoss()

optimizer = optim.AdamW(
    model.parameters(),
    lr=1e-3,
    weight_decay=1e-4
)

scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="max",
    factor=0.5,
    patience=3
)

NUM_EPOCHS = 20
GRAD_CLIP = 1.0

best_val_acc = 0.0
best_epoch = 0

history_T2 = {
    "train_loss": [],
    "train_acc": [],
    "val_loss": [],
    "val_acc": [],
    "lr": []
}

In [ ]:
# ============================================================
# T2 — TRAIN
# ============================================================

for epoch in range(1, NUM_EPOCHS + 1):

    train_loss, train_acc = run_epoch(
        model,
        thermal_T2_train_loader,
        optimizer
    )

    val_loss, val_acc = run_epoch(
        model,
        thermal_T2_val_loader,
        optimizer=None
    )

    scheduler.step(val_acc)

    current_lr = optimizer.param_groups[0]["lr"]

    history_T2["train_loss"].append(train_loss)
    history_T2["train_acc"].append(train_acc)
    history_T2["val_loss"].append(val_loss)
    history_T2["val_acc"].append(val_acc)
    history_T2["lr"].append(current_lr)

    if val_acc > best_val_acc:

        best_val_acc = val_acc
        best_epoch = epoch

        torch.save(
            {
                "epoch": epoch,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "val_accuracy": val_acc,
                "action_to_idx": action_to_idx
            },
            "thermal_T2_best.pt"
        )

        marker = " ★ BEST"

    else:
        marker = ""

    print(
        f"Epoch {epoch:02d} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Train Acc: {train_acc*100:.2f}% | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val Acc: {val_acc*100:.2f}% | "
        f"LR: {current_lr:.2e}"
        f"{marker}"
    )

print("\n" + "=" * 60)
print("T2 COMPLETE")
print("=" * 60)
print(f"Best validation accuracy: {best_val_acc*100:.2f}%")
print(f"Best epoch: {best_epoch}")

  0%|          | 0/144 [00:00<?, ?it/s]

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x788ec6b600e0>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
    if w.is_alive():
  Exception ignored in:  Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x788ec6b600e0><function _MultiProcessingDataLoaderIter.__del__ at 0x788ec6b600e0> 
 
Traceback (most recent call last):
 Traceback (most recent call last):
   File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
^^    ^    ^self._shutdown_workers()self._shutdown_workers()

^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", li

  0%|          | 0/40 [00:00<?, ?it/s]

Epoch 01 | Train Loss: 3.4171 | Train Acc: 11.85% | Val Loss: 3.5617 | Val Acc: 12.26% | LR: 1.00e-03 ★ BEST


  0%|          | 0/144 [00:00<?, ?it/s]

...

  0%|          | 0/40 [00:00<?, ?it/s]

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x788ec6b600e0>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
Exception ignored in:     self._shutdown_workers()<function _MultiProcessingDataLoaderIter.__del__ at 0x788ec6b600e0>
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers

    Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
    if w.is_alive():self._shutdown_workers()

   File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
     if w.is_alive(): 
         ^  ^^^^^^^^^^^^^^^^^^^^^
^  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
^    assert self._parent_pid == os.getpid(), 'can only test a child process'

   File "/usr/lib/pytho

Epoch 06 | Train Loss: 3.2644 | Train Acc: 14.29% | Val Loss: 3.4312 | Val Acc: 12.89% | LR: 1.00e-03


  0%|          | 0/144 [00:00<?, ?it/s]

Exception ignored in: Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x788ec6b600e0><function _MultiProcessingDataLoaderIter.__del__ at 0x788ec6b600e0>
Traceback (most recent call last):

Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
    self._shutdown_workers()  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive

  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
AssertionError:     can only test a child processif w.is_alive():
 
 Exception ignored in:  <function _MultiProcessingDataLoaderIter.__del__ at 0x788ec6b600e0>Exception ignored in:  <function _MultiProcessingDa

In [50]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model = ThermalT1(
    num_classes=40,
    frame_feature_dim=128,
    lstm_hidden=128
).to(device)

print(model)

# Parameter count
num_params = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

model_size_mb = num_params * 4 / (1024 ** 2)

print("\nDevice:", device)
print("Trainable parameters:", f"{num_params:,}")
print("Approx FP32 model size:", f"{model_size_mb:.2f} MB")

ThermalT1(
  (cnn): ThermalCNN(
    (features): Sequential(
      (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
      (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): GELU(approximate='none')
      (3): Conv2d(32, 64, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
      (4): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (5): GELU(approximate='none')
      (6): Conv2d(64, 96, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
      (7): BatchNorm2d(96, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (8): GELU(approximate='none')
      (9): Conv2d(96, 128, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
      (10): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (11): GELU(approximate='none')
      (12): AdaptiveAvgPool2d(output_size=1)
    )
    (projection): Sequential(
      (0): Flatten(start_dim=1, end_dim=

In [51]:
X, y = next(iter(thermal_train_loader))

X = X.to(device)

with torch.no_grad():
    output = model(X)

print("Input shape :", X.shape)
print("Output shape:", output.shape)

Input shape : torch.Size([16, 32, 3, 112, 112])
Output shape: torch.Size([16, 40])


In [52]:
import torch
import torch.nn as nn
import torch.optim as optim
from tqdm.auto import tqdm


criterion = nn.CrossEntropyLoss()

optimizer = optim.AdamW(
    model.parameters(),
    lr=1e-3,
    weight_decay=1e-4
)

scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="max",
    factor=0.5,
    patience=3
)

NUM_EPOCHS = 20
GRAD_CLIP = 1.0

best_val_acc = 0.0
best_epoch = 0

history = {
    "train_loss": [],
    "train_acc": [],
    "val_loss": [],
    "val_acc": [],
    "lr": []
}


def run_epoch(model, loader, optimizer=None):

    is_train = optimizer is not None

    if is_train:
        model.train()
    else:
        model.eval()

    total_loss = 0.0
    correct = 0
    total = 0

    for X, y in tqdm(loader, leave=False):

        X = X.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)

        if is_train:
            optimizer.zero_grad(set_to_none=True)

        with torch.set_grad_enabled(is_train):

            logits = model(X)

            loss = criterion(logits, y)

            if is_train:
                loss.backward()

                torch.nn.utils.clip_grad_norm_(
                    model.parameters(),
                    GRAD_CLIP
                )

                optimizer.step()

        total_loss += loss.item() * y.size(0)

        predictions = logits.argmax(dim=1)

        correct += (
            predictions == y
        ).sum().item()

        total += y.size(0)

    avg_loss = total_loss / total
    accuracy = correct / total

    return avg_loss, accuracy


for epoch in range(1, NUM_EPOCHS + 1):

    train_loss, train_acc = run_epoch(
        model,
        thermal_train_loader,
        optimizer
    )

    val_loss, val_acc = run_epoch(
        model,
        thermal_val_loader,
        optimizer=None
    )

    scheduler.step(val_acc)

    current_lr = optimizer.param_groups[0]["lr"]

    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)
    history["val_loss"].append(val_loss)
    history["val_acc"].append(val_acc)
    history["lr"].append(current_lr)

    if val_acc > best_val_acc:

        best_val_acc = val_acc
        best_epoch = epoch

        torch.save(
            {
                "epoch": epoch,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "val_accuracy": val_acc,
                "action_to_idx": action_to_idx
            },
            "thermal_T1_best.pt"
        )

        best_marker = " ★ BEST"
    else:
        best_marker = ""

    print(
        f"Epoch {epoch:02d} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Train Acc: {train_acc*100:.2f}% | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val Acc: {val_acc*100:.2f}% | "
        f"LR: {current_lr:.2e}"
        f"{best_marker}"
    )


print("\n" + "=" * 60)
print("T1 COMPLETE")
print("=" * 60)
print(f"Best validation accuracy: {best_val_acc*100:.2f}%")
print(f"Best epoch: {best_epoch}")
print("Checkpoint: thermal_T1_best.pt")

  0%|          | 0/144 [00:00<?, ?it/s]

KeyboardInterrupt: 

# A. Radar

In [ ]:
RADAR_FEATURES = [
    "x",
    "y",
    "z",
    "v",
    "snr",
    "noise"
]

RADAR_SEQUENCE_LENGTH = 64


def load_radar_trial(trial_path):
    """
    Load one Radar CSV.

    Returns:
        DataFrame with Radar detections, or None if empty.
    """

    try:
        df = pd.read_csv(trial_path)

        if df.empty:
            return None

        # Keep only required columns
        df = df[RADAR_FEATURES + ["frame"]].copy()

        # Convert numerical columns
        for col in RADAR_FEATURES:
            df[col] = pd.to_numeric(df[col], errors="coerce")

        df["frame"] = pd.to_numeric(
            df["frame"],
            errors="coerce"
        )

        # Remove invalid rows
        df = df.dropna(
            subset=RADAR_FEATURES + ["frame"]
        )

        if df.empty:
            return None

        return df

    except Exception:
        return None

In [ ]:
def radar_frame_features(df):

    grouped = []

    for frame_id, frame_df in df.groupby("frame"):

        values = frame_df[RADAR_FEATURES].to_numpy(
            dtype=np.float32
        )

        # Mean
        mean_features = values.mean(axis=0)

        # Standard deviation
        std_features = values.std(axis=0)

        # Minimum
        min_features = values.min(axis=0)

        # Maximum
        max_features = values.max(axis=0)

        # Number of detections
        object_count = np.array(
            [len(frame_df)],
            dtype=np.float32
        )

        # Frame availability
        availability = np.array(
            [1.0],
            dtype=np.float32
        )

        features = np.concatenate([
            mean_features,
            std_features,
            min_features,
            max_features,
            object_count,
            availability
        ])

        grouped.append(
            (frame_id, features)
        )

    grouped.sort(key=lambda x: x[0])

    if not grouped:
        return np.zeros(
            (RADAR_SEQUENCE_LENGTH, 26),
            dtype=np.float32
        )

    frame_features = np.stack(
        [x[1] for x in grouped],
        axis=0
    )

    return frame_features.astype(np.float32)

In [ ]:
def resample_radar_sequence(
    features,
    sequence_length=RADAR_SEQUENCE_LENGTH
):

    if len(features) == sequence_length:
        return features

    if len(features) == 1:
        return np.repeat(
            features,
            sequence_length,
            axis=0
        )

    old_positions = np.linspace(
        0,
        1,
        len(features)
    )

    new_positions = np.linspace(
        0,
        1,
        sequence_length
    )

    resampled = np.zeros(
        (sequence_length, features.shape[1]),
        dtype=np.float32
    )

    for feature_idx in range(features.shape[1]):

        resampled[:, feature_idx] = np.interp(
            new_positions,
            old_positions,
            features[:, feature_idx]
        )

    return resampled

In [ ]:
VAL_USERS = {"user8", "user9", "user23", "user24"}

radar_train_df = train_df[
    ~train_df["user"].isin(VAL_USERS)
].reset_index(drop=True)

radar_val_df = train_df[
    train_df["user"].isin(VAL_USERS)
].reset_index(drop=True)

print("Radar train samples:", len(radar_train_df))
print("Radar val samples:", len(radar_val_df))

print("\nTrain users:")
print(sorted(radar_train_df["user"].unique()))

print("\nValidation users:")
print(sorted(radar_val_df["user"].unique()))

In [ ]:
class RadarDataset(torch.utils.data.Dataset):

    def __init__(self, df, action_to_idx, radar_root):

        self.df = df.reset_index(drop=True)
        self.action_to_idx = action_to_idx
        self.radar_root = Path(radar_root)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):

        row = self.df.iloc[idx]

        # Radar trial directory:
        # Radar / action / user / trial
        trial_dir = (
            self.radar_root
            / row["action"]
            / row["user"]
            / row["trial"]
        )

        # Each trial contains one Radar CSV
        radar_files = list(trial_dir.glob("*.csv"))

        if len(radar_files) == 0:

            # No Radar file → empty Radar
            features = np.zeros(
                (RADAR_SEQUENCE_LENGTH, 26),
                dtype=np.float32
            )

        else:

            radar_df = load_radar_trial(
                radar_files[0]
            )

            if radar_df is None:

                features = np.zeros(
                    (RADAR_SEQUENCE_LENGTH, 26),
                    dtype=np.float32
                )

            else:

                features = radar_frame_features(
                    radar_df
                )

                features = resample_radar_sequence(
                    features
                )

        X = torch.tensor(
            features,
            dtype=torch.float32
        )

        y = torch.tensor(
            self.action_to_idx[row["action"]],
            dtype=torch.long
        )

        return X, y

In [ ]:
radar_train_dataset = RadarDataset(
    radar_train_df,
    action_to_idx,
    RADAR_ROOT
)

radar_val_dataset = RadarDataset(
    radar_val_df,
    action_to_idx,
    RADAR_ROOT
)

print("Train samples:", len(radar_train_dataset))
print("Val samples:", len(radar_val_dataset))

In [ ]:
def calculate_radar_normalization(
    dataset,
    max_samples=1000,
    seed=42
):

    rng = np.random.default_rng(seed)

    n = min(len(dataset), max_samples)

    indices = rng.choice(
        len(dataset),
        size=n,
        replace=False
    )

    all_features = []

    for i in indices:

        X, _ = dataset[i]

        all_features.append(
            X.numpy()
        )

    all_features = np.concatenate(
        all_features,
        axis=0
    )

    mean = all_features.mean(axis=0)
    std = all_features.std(axis=0)

    std = np.maximum(std, 1e-6)

    return (
        mean.astype(np.float32),
        std.astype(np.float32)
    )

In [ ]:
radar_mean, radar_std = calculate_radar_normalization(
    radar_train_dataset,
    max_samples=1000,
    seed=42
)

print("Mean shape:", radar_mean.shape)
print("Std shape:", radar_std.shape)

print("\nMean:")
print(radar_mean)

print("\nStd:")
print(radar_std)

In [ ]:
class NormalizedRadarDataset(RadarDataset):

    def __init__(
        self,
        df,
        action_to_idx,
        radar_root,
        mean,
        std
    ):

        super().__init__(
            df,
            action_to_idx,
            radar_root
        )

        self.mean = torch.tensor(
            mean,
            dtype=torch.float32
        )

        self.std = torch.tensor(
            std,
            dtype=torch.float32
        )

    def __getitem__(self, idx):

        X, y = super().__getitem__(idx)

        X = (X - self.mean) / self.std

        return X, y

radar_train_dataset = NormalizedRadarDataset(
    radar_train_df,
    action_to_idx,
    RADAR_ROOT,
    radar_mean,
    radar_std
)

radar_val_dataset = NormalizedRadarDataset(
    radar_val_df,
    action_to_idx,
    RADAR_ROOT,
    radar_mean,
    radar_std
)

In [ ]:
X, y = radar_train_dataset[1]

print("Shape:", X.shape)
print("Label:", y)

print("NaN:", torch.isnan(X).any().item())
print("Inf:", torch.isinf(X).any().item())

print("Min:", X.min().item())
print("Max:", X.max().item())
print("Mean:", X.mean().item())
print("Std:", X.std().item())

In [ ]:
radar_model = RadarR1(
    input_dim=26,
    hidden_dim=128,
    num_classes=40
).to(device)

print(radar_model)



In [ ]:
num_params = sum(p.numel() for p in radar_model.parameters())
model_size_mb = num_params * 4 / (1024 ** 2)

print(f"Parameters: {num_params:,}")
print(f"Approx FP32 size: {model_size_mb:.2f} MB")

In [ ]:
from torch.utils.data import DataLoader

BATCH_SIZE = 64

radar_train_loader = DataLoader(
    radar_train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

radar_val_loader = DataLoader(
    radar_val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

print("Radar train batches:", len(radar_train_loader))
print("Radar val batches:", len(radar_val_loader))

In [ ]:
X_batch, y_batch = next(iter(radar_train_loader))

print("Input :", X_batch.shape)
print("Labels:", y_batch.shape)
print("NaN   :", torch.isnan(X_batch).any().item())
print("Inf   :", torch.isinf(X_batch).any().item())

In [ ]:
import torch
import torch.nn as nn

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.AdamW(
    radar_model.parameters(),
    lr=1e-3,
    weight_decay=1e-4
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="max",
    factor=0.5,
    patience=3
)

NUM_EPOCHS = 20

best_radar_val_acc = 0.0
best_radar_epoch = 0

radar_history = []

for epoch in range(1, NUM_EPOCHS + 1):

    # =========================
    # TRAIN
    # =========================
    radar_model.train()

    train_loss = 0.0
    train_correct = 0
    train_total = 0

    for X, y in radar_train_loader:

        X = X.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)

        optimizer.zero_grad()

        logits = radar_model(X)
        loss = criterion(logits, y)

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            radar_model.parameters(),
            max_norm=1.0
        )

        optimizer.step()

        train_loss += loss.item() * X.size(0)

        predictions = logits.argmax(dim=1)
        train_correct += (predictions == y).sum().item()
        train_total += y.size(0)

    train_loss /= train_total
    train_acc = train_correct / train_total


    # =========================
    # VALIDATION
    # =========================
    radar_model.eval()

    val_loss = 0.0
    val_correct = 0
    val_total = 0

    with torch.no_grad():

        for X, y in radar_val_loader:

            X = X.to(device, non_blocking=True)
            y = y.to(device, non_blocking=True)

            logits = radar_model(X)
            loss = criterion(logits, y)

            val_loss += loss.item() * X.size(0)

            predictions = logits.argmax(dim=1)

            val_correct += (predictions == y).sum().item()
            val_total += y.size(0)

    val_loss /= val_total
    val_acc = val_correct / val_total


    # =========================
    # LR SCHEDULER
    # =========================
    scheduler.step(val_acc)

    current_lr = optimizer.param_groups[0]["lr"]

    radar_history.append({
        "epoch": epoch,
        "train_loss": train_loss,
        "train_acc": train_acc,
        "val_loss": val_loss,
        "val_acc": val_acc,
        "lr": current_lr
    })


    # =========================
    # SAVE BEST MODEL
    # =========================
    if val_acc > best_radar_val_acc:

        best_radar_val_acc = val_acc
        best_radar_epoch = epoch

        torch.save(
            {
                "model_state_dict": radar_model.state_dict(),
                "val_acc": val_acc,
                "epoch": epoch
            },
            "radar_R1_best.pt"
        )


    print(
        f"Epoch {epoch:02d}/{NUM_EPOCHS} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Train Acc: {train_acc:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val Acc: {val_acc:.4f} | "
        f"LR: {current_lr:.2e}"
    )


print("\n" + "=" * 60)
print(f"BEST R1 RADAR VALIDATION ACCURACY: {best_radar_val_acc:.4f}")
print(f"BEST EPOCH: {best_radar_epoch}")
print("=" * 60)

In [ ]:
# Check Radar availability in train/validation

def radar_availability_stats(dataset):
    empty = 0
    nonempty = 0

    for i in range(len(dataset)):
        X, _ = dataset[i]

        # Last feature is availability
        availability = X[:, -1]

        if torch.all(availability == 0):
            empty += 1
        else:
            nonempty += 1

    return empty, nonempty


train_empty, train_nonempty = radar_availability_stats(radar_train_dataset)
val_empty, val_nonempty = radar_availability_stats(radar_val_dataset)

print("RADAR AVAILABILITY")
print("-" * 40)

print(f"Train empty    : {train_empty}")
print(f"Train non-empty: {train_nonempty}")
print(f"Val empty      : {val_empty}")
print(f"Val non-empty  : {val_nonempty}")

print()
print(f"Train usable: {train_nonempty / len(radar_train_dataset):.2%}")
print(f"Val usable  : {val_nonempty / len(radar_val_dataset):.2%}")

# A. IMU

In [ ]:
IMU_ROOT = "/kaggle/input/datasets/samasiayushman/small-model-track/Training/Training/data/IMU"

print(os.path.exists(IMU_ROOT))

In [ ]:
SENSOR_ORDER = [
    "WTLA",   # Left Arm
    "WTRA",   # Right Arm
    "WTC",    # Chest
    "WTLL",   # Left Leg
    "WTRL"    # Right Leg
]
IMU_FEATURES = [
    "加速度X(g)",
    "加速度Y(g)",
    "加速度Z(g)",
    "角速度X(°/s)",
    "角速度Y(°/s)",
    "角速度Z(°/s)"
]
def load_imu_trial(trial_dir):

    sensor_data = {}

    for fname in [
        "up(LA+RA+C).csv",
        "down(LL+RL).csv"
    ]:

        path = os.path.join(trial_dir, fname)

        if not os.path.exists(path):
            continue

        df = pd.read_csv(path)

        df["时间"] = pd.to_datetime(df["时间"])

        for device in df["设备名称"].unique():

            # Identify sensor from device name
            if device.startswith("WTLA"):
                sensor = "WTLA"
            elif device.startswith("WTRA"):
                sensor = "WTRA"
            elif device.startswith("WTC"):
                sensor = "WTC"
            elif device.startswith("WTLL"):
                sensor = "WTLL"
            elif device.startswith("WTRL"):
                sensor = "WTRL"
            else:
                continue

            sensor_df = df[df["设备名称"] == device].copy()

            sensor_df = sensor_df.sort_values("时间")

            sensor_data[sensor] = sensor_df[
                ["时间"] + IMU_FEATURES
            ].reset_index(drop=True)

    return sensor_data

In [ ]:
test_trial = os.path.join(
    IMU_ROOT,
    "0_Wash_face",
    "user16",
    "1-1-1"
)

sensor_data = load_imu_trial(test_trial)

for sensor in SENSOR_ORDER:
    df = sensor_data[sensor]

    print(
        sensor,
        "shape =", df.shape,
        "start =", df["时间"].min(),
        "end =", df["时间"].max()
    )

In [ ]:
SEQUENCE_LENGTH = 64

def synchronize_imu(sensor_data, sequence_length=64):
    """
    Synchronize 5 asynchronous IMU streams onto a common timeline.

    Output:
        shape = (sequence_length, 30)
    """

    # Find the common overlapping time interval
    start_time = max(
        sensor_data[sensor]["时间"].min()
        for sensor in SENSOR_ORDER
    )

    end_time = min(
        sensor_data[sensor]["时间"].max()
        for sensor in SENSOR_ORDER
    )

    # Convert timestamps to seconds relative to start_time
    common_times = np.linspace(
        0,
        (end_time - start_time).total_seconds(),
        sequence_length
    )

    all_features = []

    for sensor in SENSOR_ORDER:

        df = sensor_data[sensor].copy()

        # Time in seconds relative to common start
        time_seconds = (
            df["时间"] - start_time
        ).dt.total_seconds().to_numpy()

        sensor_features = []

        for feature in IMU_FEATURES:

            values = df[feature].astype(float).to_numpy()

            # Interpolate onto common timeline
            interpolated = np.interp(
                common_times,
                time_seconds,
                values
            )

            sensor_features.append(interpolated)

        # (64, 6)
        sensor_features = np.stack(
            sensor_features,
            axis=1
        )

        all_features.append(sensor_features)

    # 5 × (64, 6)
    # → (64, 30)
    synchronized = np.concatenate(
        all_features,
        axis=1
    )

    return synchronized.astype(np.float32)

In [ ]:
imu_sample = synchronize_imu(sensor_data)

print("Shape:", imu_sample.shape)
print("dtype:", imu_sample.dtype)
print("min:", imu_sample.min())
print("max:", imu_sample.max())

feature_names = []

for sensor in SENSOR_ORDER:
    for feature in ["ax", "ay", "az", "gx", "gy", "gz"]:
        feature_names.append(f"{sensor}_{feature}")

print(len(feature_names))
print(feature_names)

In [ ]:
def collect_imu_trials(root_dir):
    trials = []

    for action_name in sorted(os.listdir(root_dir)):

        action_dir = os.path.join(root_dir, action_name)

        if not os.path.isdir(action_dir):
            continue

        for user_name in sorted(os.listdir(action_dir)):

            user_dir = os.path.join(action_dir, user_name)

            if not os.path.isdir(user_dir):
                continue

            for trial_name in sorted(os.listdir(user_dir)):

                trial_dir = os.path.join(user_dir, trial_name)

                if not os.path.isdir(trial_dir):
                    continue

                up_file = os.path.join(
                    trial_dir,
                    "up(LA+RA+C).csv"
                )

                down_file = os.path.join(
                    trial_dir,
                    "down(LL+RL).csv"
                )

                if os.path.exists(up_file) and os.path.exists(down_file):

                    trials.append({
                        "action": action_name,
                        "user": user_name,
                        "trial_id": f"{user_name}_{trial_name}",
                        "trial_dir": trial_dir
                    })

    return pd.DataFrame(trials)

imu_df = collect_imu_trials(IMU_ROOT)

print("Total trials:", len(imu_df))
print(imu_df.head())
print("\nActions:", imu_df["action"].nunique())
print("Users:", imu_df["user"].nunique())

In [ ]:
print(
    imu_df["action"]
    .value_counts()
    .sort_index()
)

In [ ]:
print(sorted(imu_df["user"].unique()))
print("\nTrials per user:")
print(imu_df["user"].value_counts().sort_index())

In [ ]:
# ============================================================
# IMU I1 — SUBJECT-HELD-OUT SPLIT
# ============================================================

VAL_USERS = {"user8", "user9", "user23", "user24"}

imu_train_df = imu_df[~imu_df["user"].isin(VAL_USERS)].reset_index(drop=True)
imu_val_df   = imu_df[ imu_df["user"].isin(VAL_USERS)].reset_index(drop=True)

print("Training trials:", len(imu_train_df))
print("Validation trials:", len(imu_val_df))

print("\nTraining users:")
print(sorted(imu_train_df["user"].unique()))

print("\nValidation users:")
print(sorted(imu_val_df["user"].unique()))

print("\nValidation class distribution:")
print(imu_val_df["action"].value_counts().sort_index())

In [ ]:
# ============================================================
# CLASS MAPPING
# ============================================================

ACTION_NAMES = sorted(imu_df["action"].unique())

action_to_idx = {
    action: idx
    for idx, action in enumerate(ACTION_NAMES)
}

idx_to_action = {
    idx: action
    for action, idx in action_to_idx.items()
}

print("Number of classes:", len(ACTION_NAMES))
print("\nFirst 10 classes:")
for i in range(10):
    print(i, ACTION_NAMES[i])

In [ ]:
class IMUDataset(torch.utils.data.Dataset):

    def __init__(self, df, action_to_idx):
        self.df = df.reset_index(drop=True)
        self.action_to_idx = action_to_idx

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):

        row = self.df.iloc[idx]

        sensor_data = load_imu_trial(row["trial_dir"])

        # Synchronize all 5 sensors
        features = synchronize_imu(
            sensor_data,
            sequence_length=SEQUENCE_LENGTH
        )

        X = torch.tensor(features, dtype=torch.float32)

        y = torch.tensor(
            self.action_to_idx[row["action"]],
            dtype=torch.long
        )

        return X, y

In [ ]:
imu_train_dataset = IMUDataset(
    imu_train_df,
    action_to_idx
)

X, y = imu_train_dataset[0]

print("X shape:", X.shape)
print("X dtype:", X.dtype)
print("y:", y)
print("y dtype:", y.dtype)

In [ ]:
def find_incomplete_imu_trials(df):

    incomplete = []

    for i, row in df.iterrows():

        try:
            sensor_data = load_imu_trial(row["trial_dir"])

            missing = [
                sensor
                for sensor in SENSOR_ORDER
                if sensor not in sensor_data
            ]

            if missing:
                incomplete.append({
                    "index": i,
                    "action": row["action"],
                    "user": row["user"],
                    "trial_id": row["trial_id"],
                    "missing": missing
                })

        except Exception as e:
            incomplete.append({
                "index": i,
                "action": row["action"],
                "user": row["user"],
                "trial_id": row["trial_id"],
                "missing": [f"ERROR: {e}"]
            })

    return pd.DataFrame(incomplete)


incomplete_imu = find_incomplete_imu_trials(imu_df)

print("Incomplete trials:", len(incomplete_imu))

if len(incomplete_imu) > 0:
    print(incomplete_imu.head(20).to_string(index=False))

In [ ]:
complete_imu_df = imu_df.copy()

incomplete_indices = set(incomplete_imu["index"].tolist())

complete_imu_df = complete_imu_df[
    ~complete_imu_df.index.isin(incomplete_indices)
].reset_index(drop=True)

print("Original trials:", len(imu_df))
print("Incomplete trials:", len(incomplete_imu))
print("Complete trials:", len(complete_imu_df))

In [ ]:
VAL_USERS = {"user8", "user9", "user23", "user24"}

imu_train_df = complete_imu_df[
    ~complete_imu_df["user"].isin(VAL_USERS)
].reset_index(drop=True)

imu_val_df = complete_imu_df[
    complete_imu_df["user"].isin(VAL_USERS)
].reset_index(drop=True)

print("Train:", len(imu_train_df))
print("Val:", len(imu_val_df))

In [ ]:
def calculate_imu_sensor_normalization(dataset, max_samples=1000, seed=42):

    rng = np.random.default_rng(seed)

    n = min(len(dataset), max_samples)
    indices = rng.choice(
        len(dataset),
        size=n,
        replace=False
    )

    all_features = []

    for i in indices:
        X, _ = dataset[i]
        all_features.append(X.numpy())

    # (N, 64, 30)
    all_features = np.concatenate(
        all_features,
        axis=0
    )

    # Reshape:
    # 30 = 5 sensors × 6 features
    all_features = all_features.reshape(
        -1,
        len(SENSOR_ORDER),
        len(IMU_FEATURES)
    )

    # Statistics independently for each sensor/channel
    mean = all_features.mean(axis=0)
    std = all_features.std(axis=0)

    std = np.maximum(std, 1e-6)

    return (
        mean.astype(np.float32),
        std.astype(np.float32)
    )

In [ ]:
class SensorNormalizedIMUDataset(IMUDataset):

    def __init__(
        self,
        df,
        action_to_idx,
        mean,
        std
    ):
        super().__init__(df, action_to_idx)

        self.mean = torch.tensor(
            mean,
            dtype=torch.float32
        )

        self.std = torch.tensor(
            std,
            dtype=torch.float32
        )

    def __getitem__(self, idx):

        X, y = super().__getitem__(idx)

        # (64, 30)
        X = X.reshape(
            X.shape[0],
            len(SENSOR_ORDER),
            len(IMU_FEATURES)
        )

        X = (X - self.mean) / self.std

        X = X.reshape(
            X.shape[0],
            len(SENSOR_ORDER) * len(IMU_FEATURES)
        )

        return X, y

In [ ]:
train_dataset_raw = IMUDataset(
    imu_train_df,
    action_to_idx
)

imu_sensor_mean, imu_sensor_std = calculate_imu_sensor_normalization(
    train_dataset_raw,
    max_samples=1000,
    seed=42
)

print("Mean:", imu_sensor_mean.shape)
print("Std:", imu_sensor_std.shape)

train_dataset = SensorNormalizedIMUDataset(
    imu_train_df,
    action_to_idx,
    imu_sensor_mean,
    imu_sensor_std
)

val_dataset = SensorNormalizedIMUDataset(
    imu_val_df,
    action_to_idx,
    imu_sensor_mean,
    imu_sensor_std
)

In [ ]:
train_loader = DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=64,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

In [ ]:
class I2IMUModel(nn.Module):
    def __init__(
        self,
        input_size=30,
        projection_size=128,
        hidden_size=128,
        num_layers=2,
        num_heads=4,
        num_classes=40,
        dropout=0.3
    ):
        super().__init__()

        # Input projection
        self.input_projection = nn.Sequential(
            nn.Linear(input_size, projection_size),
            nn.LayerNorm(projection_size),
            nn.GELU(),
            nn.Dropout(dropout)
        )

        # Temporal encoder
        self.lstm = nn.LSTM(
            input_size=projection_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0,
            bidirectional=True
        )

        feature_size = hidden_size * 2

        # Temporal self-attention
        self.temporal_attention = nn.MultiheadAttention(
            embed_dim=feature_size,
            num_heads=num_heads,
            dropout=dropout,
            batch_first=True
        )

        self.norm = nn.LayerNorm(feature_size)

        # Learn which frames matter
        self.frame_attention = nn.Sequential(
            nn.Linear(feature_size, 128),
            nn.Tanh(),
            nn.Linear(128, 1)
        )

        # Classifier
        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(feature_size, 128),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):

        # x: (B, 64, 30)

        x = self.input_projection(x)

        # (B, 64, 128)
        x, _ = self.lstm(x)

        # Self-attention
        attended, _ = self.temporal_attention(
            x, x, x
        )

        # Residual connection + normalization
        x = self.norm(x + attended)

        # Learn importance of each frame
        scores = self.frame_attention(x)

        # (B, 64, 1)
        weights = torch.softmax(scores, dim=1)

        # Weighted temporal pooling
        x = torch.sum(x * weights, dim=1)

        # (B, 256) → classes
        logits = self.classifier(x)

        return logits

In [ ]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model = I2IMUModel().to(device)

print("Device:", device)

total_params = sum(
    p.numel()
    for p in model.parameters()
)

print("Parameters:", total_params)
print("Approx FP32 size:", total_params * 4 / 1024**2, "MB")

In [ ]:
criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-3,
    weight_decay=1e-4
)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=20
)


def train_one_epoch(model, loader):

    model.train()

    total_loss = 0
    correct = 0
    total = 0

    for X, y in loader:

        X = X.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)

        optimizer.zero_grad()

        logits = model(X)

        loss = criterion(logits, y)

        loss.backward()

        optimizer.step()

        total_loss += loss.item() * X.size(0)

        predictions = logits.argmax(dim=1)

        correct += (predictions == y).sum().item()
        total += y.size(0)

    return total_loss / total, correct / total


@torch.no_grad()
def evaluate(model, loader):

    model.eval()

    total_loss = 0
    correct = 0
    total = 0

    for X, y in loader:

        X = X.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)

        logits = model(X)

        loss = criterion(logits, y)

        total_loss += loss.item() * X.size(0)

        predictions = logits.argmax(dim=1)

        correct += (predictions == y).sum().item()
        total += y.size(0)

    return total_loss / total, correct / total

In [ ]:
EPOCHS = 20

best_val_acc = 0.0
best_state = None
history = []

wandb.init(
    project="CIUX",
    name="IMU S6 New Model",
    config={
        "model": "BiLSTM",
        "sequence_length": 64,
        "num_keypoints": 17,
        "coordinates": 3,
        "hidden_size": 128,
        "num_layers": 2,
        "dropout": 0.3,
        "batch_size": 32,
        "learning_rate": 1e-3,
        "epochs": 50,
        "optimizer": "Adam",
    }
)

for epoch in range(1, EPOCHS + 1):

    train_loss, train_acc = train_one_epoch(
        model,
        train_loader
    )

    val_loss, val_acc = evaluate(
        model,
        val_loader
    )

    scheduler.step()

    # -------------------------
    # W&B logging
    # -------------------------
    wandb.log({
        "epoch": epoch,
        "train/loss": train_loss,
        "train/accuracy": train_acc,
        "val/loss": val_loss,
        "val/accuracy": val_acc,
        "learning_rate": optimizer.param_groups[0]["lr"]
    })

    history.append({
        "epoch": epoch,
        "train_loss": train_loss,
        "train_acc": train_acc,
        "val_loss": val_loss,
        "val_acc": val_acc
    })

    if val_acc > best_val_acc:

        best_val_acc = val_acc

        best_state = {
            k: v.cpu().clone()
            for k, v in model.state_dict().items()
        }

        # Log best result
        wandb.log({
            "best/val_accuracy": best_val_acc,
            "best/epoch": epoch
        })

    print(
        f"Epoch {epoch:02d} | "
        f"Train Acc: {train_acc:.4f} | "
        f"Val Acc: {val_acc:.4f} | "
        f"Val Loss: {val_loss:.4f}"
    )

print(
    f"\nBest I2 validation accuracy: "
    f"{best_val_acc:.4f}"
)

# B. Skeleton

In [ ]:
import torch
import torch.nn as nn


class BiLSTMAttention(nn.Module):

    def __init__(
        self,
        input_size=153,
        hidden_size=128,
        num_layers=2,
        num_classes=40,
        dropout=0.3
    ):
        super().__init__()
        self.frame_projection = nn.Sequential(

    nn.Linear(num_joints * spatial_dim, 256),
    nn.ReLU(),

    nn.Dropout(dropout),

    nn.Linear(256, spatial_dim),
    nn.ReLU()
        )

        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0,
            bidirectional=True
        )

        # 256 -> attention score
        self.attention = nn.Sequential(
            nn.Linear(hidden_size * 2, 128),
            nn.Tanh(),
            nn.Linear(128, 1)
        )

        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(hidden_size * 2, 128),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):

        # --------------------------------
        # (B, 64, 17, 6)
        # --------------------------------

        batch_size = x.size(0)

        # --------------------------------
        # Flatten joints + coordinates
        # --------------------------------

        x = x.reshape(
            batch_size,
            x.size(1),
            -1
        )

        # (B, 64, 102)

        # --------------------------------
        # BiLSTM
        # --------------------------------

        output, _ = self.lstm(x)

        # (B, 64, 256)

        # --------------------------------
        # Attention
        # --------------------------------

        scores = self.attention(output)

        # (B, 64, 1)

        weights = torch.softmax(
            scores,
            dim=1
        )

        # --------------------------------
        # Weighted temporal representation
        # --------------------------------

        context = torch.sum(
            output * weights,
            dim=1
        )

        # (B, 256)

        # --------------------------------
        # Classification
        # --------------------------------

        logits = self.classifier(context)

        return logits

In [ ]:
class S6TemporalAttentionModel(nn.Module):

    def __init__(
        self,
        input_size=204,
        projection_size=128,
        hidden_size=128,
        num_layers=2,
        num_heads=4,
        num_classes=40,
        dropout=0.3
    ):
        super().__init__()

        # -----------------------------
        # Input projection
        # -----------------------------
        self.input_projection = nn.Sequential(
            nn.Linear(input_size, projection_size),
            nn.LayerNorm(projection_size),
            nn.GELU(),
            nn.Dropout(dropout)
        )

        # -----------------------------
        # BiLSTM
        # -----------------------------
        self.lstm = nn.LSTM(
            input_size=projection_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0,
            bidirectional=True
        )

        feature_size = hidden_size * 2

        # -----------------------------
        # Multi-head temporal attention
        # -----------------------------
        self.temporal_attention = nn.MultiheadAttention(
            embed_dim=feature_size,
            num_heads=num_heads,
            dropout=dropout,
            batch_first=True
        )

        # -----------------------------
        # Residual normalization
        # -----------------------------
        self.norm = nn.LayerNorm(feature_size)

        # -----------------------------
        # LEARNED TEMPORAL WEIGHTS
        # -----------------------------
        self.frame_attention = nn.Sequential(
            nn.Linear(feature_size, 128),
            nn.Tanh(),
            nn.Linear(128, 1)
        )

        # -----------------------------
        # Classifier
        # -----------------------------
        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(feature_size, 128),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):

        # B,T,J,F
        B, T, J, F = x.shape

        # -----------------------------
        # Flatten joints
        # -----------------------------
        x = x.reshape(B, T, J * F)

        # B,T,204
        # -----------------------------
        # Project
        # -----------------------------
        x = self.input_projection(x)

        # B,T,128

        # -----------------------------
        # BiLSTM
        # -----------------------------
        x, _ = self.lstm(x)

        # B,T,256

        # -----------------------------
        # Multi-head self attention
        # -----------------------------
        attended, _ = self.temporal_attention(
            x, x, x
        )

        x = self.norm(x + attended)

        # B,T,256

        # -----------------------------
        # LEARN FRAME IMPORTANCE
        # -----------------------------
        scores = self.frame_attention(x)

        # B,T,1

        weights = torch.softmax(
            scores,
            dim=1
        )

        # -----------------------------
        # Weighted temporal representation
        # -----------------------------
        x = torch.sum(
            x * weights,
            dim=1
        )

        # B,256

        # -----------------------------
        # Classification
        # -----------------------------
        logits = self.classifier(x)

        return logits

In [ ]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model = S6TemporalAttentionModel(
    input_size=204,
    projection_size=128,
    hidden_size=128,
    num_layers=2,
    num_heads=4,
    num_classes=40,
    dropout=0.3
).to(device)

print(model)
print("Device:", device)

In [ ]:
X, y = next(iter(train_loader))

X = X.to(device)

with torch.no_grad():
    output = model(X)

print("Input :", X.shape)
print("Output:", output.shape)
X, y = next(iter(train_loader))

print("Train batch X:", X.shape)
print("Train batch y:", y.shape)

X, y = next(iter(val_loader))

print("Val batch X:", X.shape)
print("Val batch y:", y.shape)

In [ ]:
criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-3
)

In [ ]:
wandb.init(
    project="CIUX",
    name="Spatial BiLSTM S6",
    config={
        "model": "BiLSTM",
        "sequence_length": 64,
        "num_keypoints": 17,
        "coordinates": 3,
        "hidden_size": 128,
        "num_layers": 2,
        "dropout": 0.3,
        "batch_size": 32,
        "learning_rate": 1e-3,
        "epochs": 50,
        "optimizer": "Adam",
    }
)

In [ ]:
epochs = 20
best_val_accuracy = 0.0
for epoch in range(epochs):

    # ==========================================================
    # TRAIN
    # ==========================================================

    model.train()

    train_loss = 0.0
    train_correct = 0
    train_total = 0

    for X, y in train_loader:

        X = X.to(device)
        y = y.to(device)

        optimizer.zero_grad()

        output = model(X)

        loss = criterion(output, y)

        loss.backward()

        optimizer.step()

        train_loss += loss.item() * X.size(0)

        predictions = output.argmax(dim=1)

        train_correct += (predictions == y).sum().item()
        train_total += y.size(0)

    train_loss /= train_total
    train_accuracy = train_correct / train_total


    # ==========================================================
    # VALIDATION
    # ==========================================================

    model.eval()

    val_loss = 0.0
    val_correct = 0
    val_total = 0

    with torch.no_grad():

        for X, y in val_loader:

            X = X.to(device)
            y = y.to(device)

            output = model(X)

            loss = criterion(output, y)

            val_loss += loss.item() * X.size(0)

            predictions = output.argmax(dim=1)

            val_correct += (predictions == y).sum().item()
            val_total += y.size(0)

    val_loss /= val_total
    val_accuracy = val_correct / val_total
    if val_accuracy > best_val_accuracy:

        best_val_accuracy = val_accuracy

        torch.save(
            model.state_dict(),
            "best_bilstm.pt"
        )

        print(
            f"🔥 New best model: "
            f"{best_val_accuracy:.4f}"
        )


    # ==========================================================
    # PRINT
    # ==========================================================

    print(
        f"Epoch {epoch+1:02d}/{epochs} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Train Acc: {train_accuracy:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val Acc: {val_accuracy:.4f}"
    )


    # ==========================================================
    # W&B
    # ==========================================================

    wandb.log({
        "epoch": epoch + 1,

        "train/loss": train_loss,
        "train/accuracy": train_accuracy,

        "val/loss": val_loss,
        "val/accuracy": val_accuracy,

        "learning_rate": optimizer.param_groups[0]["lr"]
    })

In [ ]:
wandb.finish()

---

# Z. Inference Part

## A. Test Dataset

In [ ]:
class SkeletonTestDataset(SkeletonDataset):

    def __getitem__(self, idx):

        row = self.df.iloc[idx]

        skeleton, confidence = self.load_skeleton(
            row["skeleton"]
        )

        # --------------------------------
        # No skeleton
        # --------------------------------
        if skeleton is None:

            skeleton = np.zeros(
                (
                    self.sequence_length,
                    17,
                    3
                ),
                dtype=np.float32
            )

        # --------------------------------
        # SAME preprocessing as training
        # --------------------------------

        skeleton = self.normalize_skeleton(
            skeleton
        )

        # Velocity
        velocity = np.diff(
            skeleton,
            axis=0,
            prepend=skeleton[0:1]
        )

        # Acceleration
        acceleration = np.diff(
            velocity,
            axis=0,
            prepend=velocity[0:1]
        )

        # Bone vectors
        bone_vectors = self.get_bone_vectors(
            skeleton
        )

        # --------------------------------
        # 12 features per joint
        # --------------------------------

        features = np.concatenate(
            [
                skeleton,
                velocity,
                acceleration,
                bone_vectors,
            ],
            axis=2
        )

        # --------------------------------
        # Temporal resampling
        # --------------------------------

        features = self.temporal_resample(
            features
        )

        # --------------------------------
        # Tensor
        # --------------------------------

        X = torch.tensor(
            features,
            dtype=torch.float32
        )

        return X, row["trial_id"]

## B. Defining the Path

In [ ]:
from pathlib import Path

BASE = Path("/kaggle/input/datasets/samasiayushman/small-model-track/Testing/Testing/small_model_track_test-007/small_model_track_test")

for p in BASE.rglob("SM_test_0001"):
    print("Found:", p)

In [ ]:
TEST_ROOT = p.parent
print(TEST_ROOT)

## 3. Testing Data

In [ ]:
TEST_ROOT = Path(
    "/kaggle/input/datasets/samasiayushman/"
    "small-model-track/Testing/Testing/"
    "small_model_track_test-007/small_model_track_test"
)

test_records = []

for trial_dir in sorted(TEST_ROOT.iterdir()):

    if not trial_dir.is_dir():
        continue

    # Ignore .claude or any non-test directory
    if not trial_dir.name.startswith("SM_test_"):
        continue

    skeleton_path = trial_dir / "Skeleton"

    if not skeleton_path.is_dir():
        continue

    test_records.append({
        "trial_id": trial_dir.name,
        "path": str(trial_dir),
        "skeleton": str(skeleton_path)
    })

test_df = pd.DataFrame(test_records)

print(test_df.head())
print("Number of test trials:", len(test_df))

## 4. Calling the Dataset & Loader 

In [ ]:
test_dataset = SkeletonTestDataset(
    test_df,
    sequence_length=64
)

print("Test trials:", len(test_dataset))

In [ ]:
X, trial_id = test_dataset[0]

print("Trial:", trial_id)
print("Shape:", X.shape)
print("dtype:", X.dtype)
print("Min:", X.min().item())
print("Max:", X.max().item())

In [ ]:
test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=0,
    pin_memory=False
)

X, trial_ids = next(iter(test_loader))

print("Batch X:", X.shape)
print("Trial IDs:", trial_ids[:5])

## 5. Evaluating

In [ ]:
model.load_state_dict(torch.load("/kaggle/working/best_bilstm.pt"))
model.eval()

all_predictions = []
all_trial_ids = []

with torch.no_grad():

    for X, trial_ids in test_loader:

        X = X.to(device)

        logits = model(X)

        predictions = torch.argmax(logits, dim=1)

        all_predictions.extend(
            predictions.cpu().numpy()
        )

        all_trial_ids.extend(trial_ids)

## 6. Verifying Submission

In [ ]:
print("Number of predictions:", len(all_predictions))
print("Number of trial IDs:", len(all_trial_ids))

print("\nFirst predictions:")
for trial_id, pred in zip(
    all_trial_ids[:10],
    all_predictions[:10]
):
    print(trial_id, "->", pred)

In [ ]:
from collections import Counter

prediction_counts = Counter(all_predictions)

print("Predicted classes:")
for label, count in sorted(prediction_counts.items()):
    print(f"{label:2d}: {count}")

## 7. Submission

In [ ]:
submission = pd.DataFrame({
    "path": [
        f"small_model_track_test/{trial_id}/"
        for trial_id in all_trial_ids
    ],
    "prediction": all_predictions
})

print(submission.head())
print(submission.shape)

submission.to_csv("submission.csv",index=False)